- formulate this problem as a ML problem ie define ML objective
- choose one simple baseline model
- evaluate the baseline
- choose 2-3 other algorithms, evaluate, compare with baseline. choose one final model
- notes and summary and learnings

# data prep

In [3]:
# imports
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta


Matplotlib is building the font cache; this may take a moment.


In [5]:
df = pd.read_csv("../data/data_with_additional_features.csv")


In [19]:
import sys
sys.path.append('..')  # Add parent directory to Python path

from utils.train_test_split import chronological_train_test_split, create_ranking_test_set


In [20]:
# Split your data
train_df, val_df, test_df = chronological_train_test_split(
    df, 
    timestamp_col='timestamp',
    test_days=5, 
    val_days=5, 
    return_splits='train_val_test'
)

# Create test ground truth for evaluation
test_ground_truth = create_ranking_test_set(
    test_df,
    user_col='user_id',
    item_col='show_id',
    score_col='watch_minutes'
)

Chronological Split Summary:
  Train:      2026-06-01 00:00:00 to 2026-06-20 23:59:00 (2,612,273 rows)
  Validation: 2026-06-20 23:59:00 to 2026-06-25 23:59:00 (622,319 rows)
  Test:       2026-06-25 23:59:00 to 2026-06-30 23:59:00 (648,334 rows)

Test Set Summary:
  Users: 23,002
  Items: 5,092
  Interactions: 177,232


# training begins

## ALS

In [12]:
import pandas as pd
import numpy as np
import scipy.sparse as sparse
from implicit.als import AlternatingLeastSquares

class PhiloImplicitALSRecommender:
    """
    A math-direct implementation of Implicit ALS recommendations that handles
    internal array flipping in legacy library versions.
    """
    def __init__(self, factors=64, regularization=0.1, iterations=15):
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.model = None
        
        self.user_to_idx = {}
        self.idx_to_show = {}
        self.user_item_matrix = None
        self.train_seen_shows = {} # Maps internal user_code -> set of internal show_codes

    def fit(self, train_df):
        print("Transforming interaction logs into Strict Sparse Matrices...")
        
        working_df = train_df.copy()
        working_df['user_id'] = working_df['user_id'].astype('category')
        working_df['show_id'] = working_df['show_id'].astype('category')
        
        user_codes = working_df['user_id'].cat.codes.values
        show_codes = working_df['show_id'].cat.codes.values
        scores = working_df['implicit_interaction_score'].values.astype(np.float32)
        
        self.user_to_idx = dict(zip(working_df['user_id'], user_codes))
        self.idx_to_show = dict(zip(show_codes, working_df['show_id']))
        
        # Directly map internal integer user codes to internal integer show codes
        temp_df = pd.DataFrame({'u': user_codes, 's': show_codes})
        self.train_seen_shows = temp_df.groupby('u')['s'].apply(set).to_dict()
        
        num_users = int(user_codes.max() + 1)
        num_shows = int(show_codes.max() + 1)
        print(f"  - Matrix Layout: {num_users} users x {num_shows} shows")
        
        self.user_item_matrix = sparse.csr_matrix(
            (scores, (user_codes, show_codes)), 
            shape=(num_users, num_shows)
        )
        
        print(f"Training Implicit ALS (Factors: {self.factors}, Iterations: {self.iterations})...")
        self.model = AlternatingLeastSquares(
            factors=self.factors, 
            regularization=self.regularization, 
            iterations=self.iterations, 
            random_state=42
        )
        
        # Train natively on item-user orientation
        self.model.fit(self.user_item_matrix.T, show_progress=True)
        print("  - [SUCCESS] Implicit ALS Factorization complete.")

    def predict(self, user_id, top_n=10, filter_seen=True):
        """Generates recommendations via robust dot-product scoring array checks."""
        if user_id not in self.user_to_idx:
            return []
            
        user_idx = self.user_to_idx[user_id]
        
        # --- FIXED FOR TRANSPOSED LEGACY VERSIONS ---
        # Because we train on .T, item_factors holds users, and user_factors holds items!
        user_factor = self.model.item_factors[user_idx]  # Extracting user from size 97832 matrix
        item_factors = self.model.user_factors          # Extracting items from size 12240 matrix
        
        # Matrix multiply user vector with all item factors to get score array
        scores = np.dot(item_factors, user_factor)
        
        # Mask out shows the user has already watched if filter_seen is active
        if filter_seen and user_idx in self.train_seen_shows:
            seen_items = list(self.train_seen_shows[user_idx])
            valid_seen_items = [idx for idx in seen_items if idx < len(scores)]
            scores[valid_seen_items] = -np.inf 
            
        # Get indices of the top highest scores sorting from best to worst
        top_indices = np.argsort(scores)[::-1][:top_n]
        
        # Map raw internal indices back to original Philo Show IDs safely
        recommended_shows = []
        for idx in top_indices:
            if scores[idx] == -np.inf:
                continue
            show_id = self.idx_to_show.get(idx, None)
            if show_id is not None:
                recommended_shows.append(show_id)
                
        return recommended_shows[:top_n]

In [15]:
import pandas as pd
import numpy as np
import scipy.sparse as sparse
from implicit.als import AlternatingLeastSquares
from sklearn.metrics.pairwise import cosine_similarity


# ==============================================================================
# 3. HIGH-SPEED PRODUCTION EVALUATION ENGINE
# ==============================================================================
class PhiloEvaluationEngine:
    """
    Computes professional information retrieval benchmarks (Recall@K, MRR@K)
    safely and rapidly without memory leaks.
    """
    @staticmethod
    def evaluate_model(model_instance, test_ground_truth, train_seen_shows, top_n=10):
        print(f"Beginning Evaluation Pipeline (Top N = {top_n})...")
        
        recalls = []
        mrrs = []
        
        for idx, row in test_ground_truth.iterrows():
            user_id = row['user_id']
            actual_shows = row['actual_shows_watched'] # Set of true future shows
            
            # 1. Generate recommendations from the passed model instance
            # Try to pass historical maps if the model is content/popularity-based
            try:
                predicted_shows = model_instance.predict(user_id, top_n=top_n, filter_seen=True)
            except TypeError:
                predicted_shows = model_instance.predict(user_id, train_seen_shows=train_seen_shows, top_n=top_n)
                
            if not predicted_shows:
                continue
                
            # 2. Compute Recall@N
            hits = [show for show in predicted_shows if show in actual_shows]
            recall_score = len(hits) / min(len(actual_shows), top_n) if len(actual_shows) > 0 else 0
            recalls.append(recall_score)
            
            # 3. Compute MRR@N (Mean Reciprocal Rank)
            mrr_score = 0
            for rank_idx, show in enumerate(predicted_shows, 1):
                if show in actual_shows:
                    mrr_score = 1.0 / rank_idx
                    break # Stop at first true hit position
            mrrs.append(mrr_score)
            
        mean_recall = np.mean(recalls) * 100
        mean_mrr = np.mean(mrrs)
        
        print("\n==========================================================")
        print("                  EVALUATION REPORT                       ")
        print("==========================================================")
        print(f"  - Target Test Evaluation Users Count : {len(recalls):,}")
        print(f"  - Average Recall@{top_n}                : {mean_recall:.2f}%")
        print(f"  - Mean Reciprocal Rank (MRR@{top_n})   : {mean_mrr:.4f}")
        print("==========================================================\n")
        
        return mean_recall, mean_mrr

In [16]:
# --- 1. Prepare historical lookups required for clean recommendation processing ---

# --- 1. Prepare historical lookups ---
print("Mapping training interaction lookup states...")
train_seen_shows = train_features.groupby('user_id')['show_id'].apply(set).to_dict()

# ==============================================================================
# OPTIMIZATION STEP: Add the Log-Scale transformation right here!
# ==============================================================================
print("Dampening watch-time outliers via Log-Scaling...")
train_features['implicit_interaction_score'] = np.log1p(train_features['implicit_interaction_score'])
# --- 2. Initialize and Train the Implicit ALS Model ---
# This version handles internal index matching securely via raw numpy arrays
als_recommender = PhiloImplicitALSRecommender(factors=64, regularization=0.1, iterations=15)
als_recommender.fit(train_features)


# --- 4. Evaluate Collaborative Filtering Engine ---
print("\n[RUNNING EVALUATION ON IMPLICIT ALS MODEL]")
# We safely pass train_seen_shows here because the evaluation engine's 
# internal try/except handles the parameter filtering for ALS.
als_recall, als_mrr = PhiloEvaluationEngine.evaluate_model(
    model_instance=als_recommender,
    test_ground_truth=test_ground_truth,
    train_seen_shows=train_seen_shows,
    top_n=10
)



Mapping training interaction lookup states...
Dampening watch-time outliers via Log-Scaling...
Transforming interaction logs into Strict Sparse Matrices...
  - Matrix Layout: 97832 users x 12240 shows
Training Implicit ALS (Factors: 64, Iterations: 15)...


/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/Users/smalik/Documents/lucky/p3/lib/python3.9/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.007611989974975586 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

  - [SUCCESS] Implicit ALS Factorization complete.

[RUNNING EVALUATION ON IMPLICIT ALS MODEL]
Beginning Evaluation Pipeline (Top N = 10)...


TypeError: 'in <string>' requires string as left operand, not int

In [35]:
import pickle

# Define the filename where you want to save the model
model_filename = "../models/philo_als_recommender.pkl"

# Open a file in write-binary ('wb') mode and dump the model instance
with open(model_filename, "wb") as file:
    pickle.dump(als_recommender, file)

print(f"[SUCCESS] Model successfully serialized and saved to {model_filename}")

[SUCCESS] Model successfully serialized and saved to ../models/philo_als_recommender.pkl


In [36]:
# Open the file in read-binary ('rb') mode and load it
with open(model_filename, "rb") as file:
    als_recommender_1 = pickle.load(file)

print("[SUCCESS] Model successfully reloaded into memory.")

[SUCCESS] Model successfully reloaded into memory.


# ALS + LGB

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb



# ==============================================================================
print("Extracting network profiles and item popularity states...")

# 1. Show-Level Metrics (Capturing Global Trends)
show_features = train_features.groupby('show_id').agg(
    show_global_popularity=('user_id', 'count'),
    show_total_watch_time=('implicit_interaction_score', 'sum')
).reset_index()

# Log scale continuous features to eliminate extreme power-law tail variance
show_features['show_global_popularity_log'] = np.log1p(show_features['show_global_popularity'])
show_features['show_total_watch_time_log'] = np.log1p(show_features['show_total_watch_time'])

# 2. User-Level Metrics (Capturing General Platform Engagement)
user_features = train_features.groupby('user_id').agg(
    user_total_interactions=('show_id', 'count'),
    user_avg_interaction_score=('implicit_interaction_score', 'mean')
).reset_index()


# ==============================================================================
# STEP 2: CONSTRUCT INTENT LABELS & THE INTEGRATED POINTWISE MATRIX
# ==============================================================================
print("Building explicit training matrix...")
ranking_train_df = train_features.copy()

# Discretize continuous watch-times into ordinal relevance grades (0, 1, 2)
# LightGBM LambdaRank strictly requires categorical integers for pairwise loss calculations
q50 = ranking_train_df['implicit_interaction_score'].median()
q90 = ranking_train_df['implicit_interaction_score'].quantile(0.90)

ranking_train_df['relevance'] = 0
ranking_train_df.loc[ranking_train_df['implicit_interaction_score'] > q50, 'relevance'] = 1
ranking_train_df.loc[ranking_train_df['implicit_interaction_score'] > q90, 'relevance'] = 2

# Map feature dimensions back into base historical logs
ranking_train_df = ranking_train_df.merge(show_features, on='show_id', how='left')
ranking_train_df = ranking_train_df.merge(user_features, on='user_id', how='left')

# CRITICAL LIGHTGBM RULE: Rows must be strictly grouped/sorted by user_id. 
# LambdaRank loops through the dataset sequentially and reads contiguous blocks.
ranking_train_df = ranking_train_df.sort_values(by='user_id').reset_index(drop=True)

# ==============================================================================
# PHASE 1: CHRONOLOGICAL OUT-OF-FOLD (OOF) MATRIX SPLITTING
# ==============================================================================
max_time = df['timestamp'].max()
test_split = max_time - pd.Timedelta(days=5)       # Final 5 days for testing
lgb_split = test_split - pd.Timedelta(days=5)      # Middle 5 days for LightGBM features

print(f"Timeline 1 (ALS Fit Window)      : {df['timestamp'].min()} to {lgb_split}")
print(f"Timeline 2 (LightGBM Fit Window): {lgb_split} to {test_split}")
print(f"Timeline 3 (Production Test Box): {test_split} to {max_time}")

# Slicing log frames
logs_als = df[df['timestamp'] < lgb_split].copy()
logs_lgb = df[(df['timestamp'] >= lgb_split) & (df['timestamp'] < test_split)].copy()
logs_test = df[df['timestamp'] >= test_split].copy()

# Weight Mapping
asset_multipliers = {'LOOKBACK': 2.0, 'VOD': 1.5, 'RECORDING': 1.2, 'CHANNEL': 1.0}
for frame in [logs_als, logs_lgb]:
    frame['calculated_score'] = frame['watch_minutes'] * frame['asset_type'].map(asset_multipliers).fillna(1.0)

# Build features
als_train_features = logs_als.groupby(['user_id', 'show_id']).agg(
    total_watch_minutes=('watch_minutes', 'sum'), implicit_interaction_score=('calculated_score', 'sum')
).reset_index()
als_train_features = als_train_features[als_train_features['total_watch_minutes'] >= 2]

lgb_train_features = logs_lgb.groupby(['user_id', 'show_id']).agg(
    total_watch_minutes=('watch_minutes', 'sum'), implicit_interaction_score=('calculated_score', 'sum')
).reset_index()

test_ground_truth = logs_test.groupby('user_id')['show_id'].apply(set).reset_index()
test_ground_truth.columns = ['user_id', 'actual_shows_watched']
test_ground_truth = test_ground_truth[test_ground_truth['user_id'].isin(als_train_features['user_id'].unique())]

# ==============================================================================
# PHASE 2: FIT INTERMEDIATE RETRIEVAL MODEL
# ==============================================================================
print("\nFitting Stage 1 ALS Recommender on Timeline 1 data...")
# --- EXECUTE YOUR ALS MODEL FIT HERE ---
# als_recommender.fit(als_train_features) 
# ----------------------------------------

# ==============================================================================
# PHASE 3: LEAKAGE-FREE MATRIX GENERATION FOR LIGHTGBM
# ==============================================================================
user_factors = als_recommender.model.item_factors  
item_factors = als_recommender.model.user_factors  
show_to_idx = {v: k for k, v in als_recommender.idx_to_show.items()}

unique_lgb_users = lgb_train_features['user_id'].unique()
all_shows_pool = show_features['show_id'].unique()
global_top_shows = show_features.sort_values(by='show_global_popularity_log', ascending=False)['show_id'].head(100).values

balanced_rows = []

print("Extracting Out-of-Fold features to eliminate distribution bias...")
for user_id in unique_lgb_users:
    if user_id not in als_recommender.user_to_idx:
        continue
    user_idx = als_recommender.user_to_idx[user_id]
    if user_idx >= len(user_factors):
        continue
    user_vector = user_factors[user_idx]
    
    # Target values from Timeline 2 (which the ALS vectors have NEVER seen!)
    pos_df = lgb_train_features[lgb_train_features['user_id'] == user_id]
    seen_shows = set(pos_df['show_id'])
    
    q50_user = pos_df['implicit_interaction_score'].median()
    for _, row in pos_df.iterrows():
        show_id = row['show_id']
        show_idx = show_to_idx.get(show_id, None)
        
        # This score now matches the distribution LightGBM will encounter at test time!
        als_score = np.dot(user_vector, item_factors[show_idx]) if show_idx is not None else 0.0
        rel_score = 2 if row['implicit_interaction_score'] >= q50_user else 1
        
        balanced_rows.append({
            'user_id': user_id, 'show_id': show_id, 'relevance': rel_score,
            'als_personalization_score': als_score
        })
        
    # Inject background popular items for contrastive balancing
    unseen_popular = [s for s in global_top_shows if s not in seen_shows]
    if unseen_popular:
        neg_samples = np.random.choice(unseen_popular, size=min(5, len(unseen_popular)), replace=False)
        for neg_show_id in neg_samples:
            show_idx = show_to_idx.get(neg_show_id, None)
            als_score = np.dot(user_vector, item_factors[show_idx]) if show_idx is not None else 0.0
            
            balanced_rows.append({
                'user_id': user_id, 'show_id': neg_show_id, 'relevance': 0,
                'als_personalization_score': als_score
            })

ranking_train_df = pd.DataFrame(balanced_rows)
ranking_train_df = ranking_train_df.merge(show_features, on='show_id', how='left').fillna(0)
ranking_train_df = ranking_train_df.merge(user_features, on='user_id', how='left').fillna(0)

# Feature Transforms
ranking_train_df['popularity_bias_resistance'] = ranking_train_df['user_avg_interaction_score'] - ranking_train_df['show_global_popularity_log']
ranking_train_df['user_show_interaction_ratio'] = ranking_train_df['user_total_interactions'] / (ranking_train_df['show_global_popularity_log'] + 1)
ranking_train_df['als_to_popularity_ratio'] = ranking_train_df['als_personalization_score'] / (ranking_train_df['show_global_popularity_log'] + 1)
ranking_train_df['pop_dominance_boost'] = ranking_train_df['show_global_popularity_log'] * ranking_train_df['als_personalization_score']

extended_feature_cols = [
    'als_personalization_score', 'show_global_popularity_log', 'show_total_watch_time_log',
    'user_total_interactions', 'user_avg_interaction_score', 'popularity_bias_resistance',      
    'user_show_interaction_ratio', 'als_to_popularity_ratio', 'pop_dominance_boost'
]

ranking_train_df = ranking_train_df.sort_values(by='user_id').reset_index(drop=True)
train_group_counts = ranking_train_df.groupby('user_id').size().values

# ==============================================================================
# PHASE 4: TRAIN HIGH-GENERALIZATION RANKER
# ==============================================================================
X_train_ext = ranking_train_df[extended_feature_cols]
y_train_ext = ranking_train_df['relevance']
train_dataset_ext = lgb.Dataset(X_train_ext, label=y_train_ext, group=train_group_counts)

beating_baseline_params = {
    'objective': 'lambdarank', 'metric': 'map', 'map_eval_at': [10],
    'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 63, 'verbosity': -1, 'seed': 42
}
extended_ranker_model = lgb.train(params=beating_baseline_params, train_set=train_dataset_ext, num_boost_round=150)



In [ ]:


# Define the filename where you want to save the model
model_filename = "../models/philo_als_plus_lgb_recommender.pkl"

# Open a file in write-binary ('wb') mode and dump the model instance
with open(model_filename, "wb") as file:
    pickle.dump(als_recommender, file)

print(f"[SUCCESS] Model successfully serialized and saved to {model_filename}")

In [ ]:
# ==============================================================================
# PHASE 4: DEFINE MULTI-CHANNEL EVALUATION ENGINE CLASS
# ==============================================================================
class PhiloBaselineBreakerEngine:
    """
    Two-Stage Hybrid Engine utilizing Multi-Channel Sourcing (ALS + Global Popularity Injection)
    to beat dominant baseline recall benchmarks.
    """
    @staticmethod
    def evaluate_pipeline(als_model, lgb_ranker, test_ground_truth, train_seen_shows, 
                          show_features, user_features, feature_cols, global_top_shows, top_n=10):
        print(f"Beginning Multi-Channel Hybrid Pipeline Execution...")
        
        recalls = []
        mrrs = []
        
        show_feat_dict = show_features.set_index('show_id')[['show_global_popularity_log', 'show_total_watch_time_log']].to_dict('index')
        user_feat_dict = user_features.set_index('user_id')[['user_total_interactions', 'user_avg_interaction_score']].to_dict('index')
        show_to_idx = {v: k for k, v in als_model.idx_to_show.items()}
        
        avg_user_interactions = user_features['user_total_interactions'].mean()
        avg_user_score = user_features['user_avg_interaction_score'].mean()
        
        user_factors = als_model.model.item_factors
        item_factors = als_model.model.user_factors
        
        test_sample = test_ground_truth.sample(n=min(3000, len(test_ground_truth)), random_state=42)
        
        for idx, row in test_sample.iterrows():
            user_id = row['user_id']
            actual_shows = row['actual_shows_watched']
            
            # CHANNEL 1: Personalized Collaborative Candidates
            als_candidates = als_model.predict(user_id, top_n=300, filter_seen=True)
            if not als_candidates or user_id not in als_model.user_to_idx:
                continue
                
            # CHANNEL 2: Inject Global Baseline Blockbusters directly into the candidate mix
            combined_candidates = list(dict.fromkeys(als_candidates + list(global_top_shows[:150])))
            
            user_idx = als_model.user_to_idx[user_id]
            if user_idx >= len(user_factors):
                continue
            user_vector = user_factors[user_idx]
            
            u_feats = user_feat_dict.get(user_id, {'user_total_interactions': avg_user_interactions, 'user_avg_interaction_score': avg_user_score})
            
            candidate_rows = []
            for show_id in combined_candidates:
                show_idx = show_to_idx.get(show_id, None)
                
                if show_idx is not None and show_idx < len(item_factors):
                    als_score = np.dot(user_vector, item_factors[show_idx])
                else:
                    als_score = 0.0 
                    
                s_feats = show_feat_dict.get(show_id, {'show_global_popularity_log': 0, 'show_total_watch_time_log': 0})
                
                pop_bias_res = u_feats['user_avg_interaction_score'] - s_feats['show_global_popularity_log']
                user_show_ratio = u_feats['user_total_interactions'] / (s_feats['show_global_popularity_log'] + 1)
                als_pop_ratio = als_score / (s_feats['show_global_popularity_log'] + 1)
                pop_boost = s_feats['show_global_popularity_log'] * als_score
                
                candidate_rows.append([
                    als_score,
                    s_feats['show_global_popularity_log'],
                    s_feats['show_total_watch_time_log'],
                    u_feats['user_total_interactions'],
                    u_feats['user_avg_interaction_score'],
                    pop_bias_res,    
                    user_show_ratio,  
                    als_pop_ratio,
                    pop_boost
                ])
                
            X_candidates = pd.DataFrame(candidate_rows, columns=feature_cols)
            predicted_scores = lgb_ranker.predict(X_candidates)
            
            top_indices = np.argsort(predicted_scores)[::-1][:top_n]
            final_predictions = [combined_candidates[i] for i in top_indices]
            
            hits = [show for show in final_predictions if show in actual_shows]
            recall_score = len(hits) / min(len(actual_shows), top_n) if len(actual_shows) > 0 else 0
            recalls.append(recall_score)
            
            mrr_score = 0
            for rank_idx, show in enumerate(final_predictions, 1):
                if show in actual_shows:
                    mrr_score = 1.0 / rank_idx
                    break
            mrrs.append(mrr_score)
            
        mean_recall = np.mean(recalls) * 100
        mean_mrr = np.mean(mrrs)
        
        print("\n==========================================================")
        print("          HYBRID BASELINE-BREAKER FINAL METRICS           ")
        print("==========================================================")
        print(f"  - Final Recall@10                      : {mean_recall:.2f}%")
        print(f"  - Final Mean Reciprocal Rank (MRR@10)  : {mean_mrr:.4f}")
        print("==========================================================\n")
        
        return mean_recall, mean_mrr

# ==============================================================================
# PHASE 5: BASELINE BREAKER EXECUTION CALL
# ==============================================================================
pipeline_recall, pipeline_mrr = PhiloBaselineBreakerEngine.evaluate_pipeline(
    als_model=als_recommender,          
    lgb_ranker=extended_ranker_model,  
    test_ground_truth=test_ground_truth,
    train_seen_shows=als_train_features, 
    show_features=show_features,
    user_features=user_features,
    feature_cols=extended_feature_cols,
    global_top_shows=global_top_shows 
)

## popularity + lgb

In [39]:
import numpy as np
import pandas as pd
import lightgbm as lgb

print("==========================================================")
print(" PHASE 1: CHRONOLOGICAL SPLITTING & POPULARITY GENERATION ")
print("==========================================================")

max_time = df['timestamp'].max()
test_split = max_time - pd.Timedelta(days=5)       # Final 5 days for testing
lgb_split = test_split - pd.Timedelta(days=5)      # Middle 5 days for LightGBM features

# Slicing log frames chronologically to prevent data leakage
logs_train_pool = df[df['timestamp'] < lgb_split].copy()
logs_lgb = df[(df['timestamp'] >= lgb_split) & (df['timestamp'] < test_split)].copy()
logs_test = df[df['timestamp'] >= test_split].copy()

# Calculate the Global Popularity Pool from the training logs
# This replaces the ALS candidate generator entirely
global_popularity = logs_train_pool.groupby('show_id').size().reset_index(name='global_views')
global_popularity['show_global_popularity_log'] = np.log1p(global_popularity['global_views'])

# Define our Stage 1 Retrieval Pool (Top 500 most popular shows)
top_500_popularity_pool = global_popularity.sort_values(by='show_global_popularity_log', ascending=False)['show_id'].head(500).values

# Filter show_features and user_features to ensure alignment
show_features_clean = show_features.drop(columns=['show_global_popularity_log'], errors='ignore').merge(global_popularity, on='show_id', how='left').fillna(0)

lgb_train_features = logs_lgb.groupby(['user_id', 'show_id']).size().reset_index(name='user_show_interactions')

test_ground_truth = logs_test.groupby('user_id')['show_id'].apply(set).reset_index()
test_ground_truth.columns = ['user_id', 'actual_shows_watched']

# ==============================================================================
# PHASE 2: MATRIX GENERATION (POPULARITY CANDIDATES + USER FEATURES)
# ==============================================================================
unique_lgb_users = lgb_train_features['user_id'].unique()
balanced_rows = []

print("Building ranking matrix by passing the popularity pool through LightGBM...")
for user_id in unique_lgb_users:
    pos_df = lgb_train_features[lgb_train_features['user_id'] == user_id]
    seen_shows = set(pos_df['show_id'])
    
    # 1. Positive Rows (Items from the popularity pool the user actually watched in this window)
    for _, row in pos_df.iterrows():
        show_id = row['show_id']
        if show_id in top_500_popularity_pool:
            balanced_rows.append({
                'user_id': user_id, 'show_id': show_id, 'relevance': 1
            })
            
    # 2. Negative Rows (Items from the popularity pool the user skipped / did not watch)
    unseen_popular = [s for s in top_500_popularity_pool if s not in seen_shows]
    if unseen_popular:
        neg_samples = np.random.choice(unseen_popular, size=min(10, len(unseen_popular)), replace=False)
        for neg_show_id in neg_samples:
            balanced_rows.append({
                'user_id': user_id, 'show_id': neg_show_id, 'relevance': 0
            })

ranking_train_df = pd.DataFrame(balanced_rows)
ranking_train_df = ranking_train_df.merge(show_features_clean, on='show_id', how='left').fillna(0)
ranking_train_df = ranking_train_df.merge(user_features, on='user_id', how='left').fillna(0)

# ==============================================================================
# PHASE 3: FEATURE ENGINEERING (WITHOUT ALS SCORES)
# ==============================================================================
ranking_train_df['popularity_bias_resistance'] = (
    ranking_train_df['user_avg_interaction_score'] - ranking_train_df['show_global_popularity_log']
)
ranking_train_df['user_show_interaction_ratio'] = (
    ranking_train_df['user_total_interactions'] / (ranking_train_df['show_global_popularity_log'] + 1)
)

# Feature footprint tailored for pure metadata interaction ranking
feature_cols = [
    'show_global_popularity_log', 
    'show_total_watch_time_log',
    'user_total_interactions', 
    'user_avg_interaction_score',
    'popularity_bias_resistance',      
    'user_show_interaction_ratio'     
]

ranking_train_df = ranking_train_df.sort_values(by='user_id').reset_index(drop=True)
train_group_counts = ranking_train_df.groupby('user_id').size().values

# Train the pure Popularity-Ranker Tree
X_train = ranking_train_df[feature_cols]
y_train = ranking_train_df['relevance']
train_dataset = lgb.Dataset(X_train, label=y_train, group=train_group_counts)

params = {
    'objective': 'lambdarank', 'metric': 'map', 'map_eval_at': [10],
    'learning_rate': 0.05, 'max_depth': 7, 'num_leaves': 63, 'verbosity': -1, 'seed': 42
}
pure_lgb_model = lgb.train(params=params, train_set=train_dataset, num_boost_round=150)

# ==============================================================================
# PHASE 4: EVALUATION ENGINE
# ==============================================================================
class PhiloPurePopularityLgbEngine:
    @staticmethod
    def evaluate_pipeline(lgb_ranker, test_ground_truth, show_features, user_features, feature_cols, top_popularity_pool, top_n=10):
        print(f"Beginning Pure Popularity + LightGBM Evaluation Loop...")
        recalls = []
        mrrs = []
        
        show_feat_dict = show_features.set_index('show_id')[['show_global_popularity_log', 'show_total_watch_time_log']].to_dict('index')
        user_feat_dict = user_features.set_index('user_id')[['user_total_interactions', 'user_avg_interaction_score']].to_dict('index')
        
        avg_user_interactions = user_features['user_total_interactions'].mean()
        avg_user_score = user_features['user_avg_interaction_score'].mean()
        
        test_sample = test_ground_truth.sample(n=min(3000, len(test_ground_truth)), random_state=42)
        
        for idx, row in test_sample.iterrows():
            user_id = row['user_id']
            actual_shows = row['actual_shows_watched']
            u_feats = user_feat_dict.get(user_id, {'user_total_interactions': avg_user_interactions, 'user_avg_interaction_score': avg_user_score})
            
            candidate_rows = []
            for show_id in top_popularity_pool:
                s_feats = show_feat_dict.get(show_id, {'show_global_popularity_log': 0, 'show_total_watch_time_log': 0})
                
                pop_bias_res = u_feats['user_avg_interaction_score'] - s_feats['show_global_popularity_log']
                user_show_ratio = u_feats['user_total_interactions'] / (s_feats['show_global_popularity_log'] + 1)
                
                candidate_rows.append([
                    s_feats['show_global_popularity_log'],
                    s_feats['show_total_watch_time_log'],
                    u_feats['user_total_interactions'],
                    u_feats['user_avg_interaction_score'],
                    pop_bias_res,    
                    user_show_ratio
                ])
                
            X_candidates = pd.DataFrame(candidate_rows, columns=feature_cols)
            predicted_scores = lgb_ranker.predict(X_candidates)
            
            top_indices = np.argsort(predicted_scores)[::-1][:top_n]
            final_predictions = [top_popularity_pool[i] for i in top_indices]
            
            hits = [show for show in final_predictions if show in actual_shows]
            recall_score = len(hits) / min(len(actual_shows), top_n) if len(actual_shows) > 0 else 0
            recalls.append(recall_score)
            
            mrr_score = 0
            for rank_idx, show in enumerate(final_predictions, 1):
                if show in actual_shows:
                    mrr_score = 1.0 / rank_idx
                    break
            mrrs.append(mrr_score)
            
        print("\n==========================================================")
        print("          PURE POPULARITY + LIGHTGBM BENCHMARKS           ")
        print("==========================================================")
        print(f"  - Final Recall@10                      : {np.mean(recalls)*100:.2f}%")
        print(f"  - Final Mean Reciprocal Rank (MRR@10)  : {np.mean(mrrs):.4f}")
        print("==========================================================\n")

# Run Evaluation
PhiloPurePopularityLgbEngine.evaluate_pipeline(
    lgb_ranker=pure_lgb_model, test_ground_truth=test_ground_truth,
    show_features=show_features_clean, user_features=user_features,
    feature_cols=feature_cols, top_popularity_pool=top_500_popularity_pool
)

 PHASE 1: CHRONOLOGICAL SPLITTING & POPULARITY GENERATION 
Building ranking matrix by passing the popularity pool through LightGBM...
Beginning Pure Popularity + LightGBM Evaluation Loop...

          PURE POPULARITY + LIGHTGBM BENCHMARKS           
  - Final Recall@10                      : 20.60%
  - Final Mean Reciprocal Rank (MRR@10)  : 0.2091

